In [ ]:
import pandas as pd
import random
import os
import chardet

''''
with open("/workspace/dataset/train/label/train_anger.json", "rb") as f:
    result = chardet.detect(f.read())
    print(result)
'''
def upload_category(set, category):  
    # 데이터 불러오기
    data = pd.read_json(f"/workspace/dataset/{set}/label/{set}_{category}.json", encoding='EUC-KR')

    print(data['faceExp_uploader'].unique()[0])

    # 샘플링 - 층화 샘플링 (감정과 배경 기준)
    sampled_data = data.groupby(["faceExp_uploader", "bg_uploader"]).apply(
        lambda x: x.sample(frac=0.3, random_state=42)
    ).reset_index(drop=True)
    
    return sampled_data

# 검증 함수
def validate_metadata(row, set, category):
    errors = []
    # 파일 존재 확인
    if not os.path.exists(f"/workspace/dataset/{set}/image/{category}/{row['filename']}"):
        errors.append("파일 없음")
    # 바운딩 박스 검증
    boxes = row['annot_A']['boxes']
    if boxes['minX'] > boxes['maxX'] or boxes['minY'] > boxes['maxY']:
        errors.append("바운딩 박스 오류")
    # 라벨 범주 검증
    if row['faceExp_uploader'] not in ["분노", "기쁨", "슬픔", "당황"]:
        errors.append("감정 라벨 오류")
    if row['bg_uploader'] not in ['교통/이동수단(엘리베이터 포함)', '문화재 및 유적지', '오락/공연시설', '공공시설/종교/의료시설', '상업시설/점포/시장',
 '숙박 및 거주공간', '행사/사무공간', '도심 환경', '실외 자연환경', '스포츠 관람 및 레저시설']:
        errors.append("배경 라벨 오류")
    return errors

def validate_data(set):
    print(f"{set} 세트 샘플링 검증")
    print()
    for i in range(4):
        if i == 1:
            emotion = 'happy'
        elif i == 2:
            emotion = 'panic'
        elif i == 3:
            emotion = 'sadness'
        else: 
            emotion = 'anger'
        
        sampled_data = upload_category(set, emotion)
        # 검증 실행
        sampled_data["errors"] = sampled_data.apply(validate_metadata, axis=1, set = set, category = emotion)

        # 검증 결과 요약
        invalid_rows = sampled_data[sampled_data["errors"].apply(len) > 0]
        print(f"검증 실패: {len(invalid_rows)}개")
        print(invalid_rows[["filename", "errors"]])
        print()

validate_data("train")

train 세트 샘플링 검증

분노
검증 실패: 0개
Empty DataFrame
Columns: [filename, errors]
Index: []

기쁨
검증 실패: 0개
Empty DataFrame
Columns: [filename, errors]
Index: []



/tmp/ipykernel_18013/1386583775.py:18: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled_data = data.groupby(["faceExp_uploader", "bg_uploader"]).apply(
/tmp/ipykernel_18013/1386583775.py:18: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled_data = data.groupby(["faceExp_uploader", "bg_uploader"]).apply(
/tmp/ipykernel_18013/1386583775.py:18: DeprecationWarning: DataFrameGroupBy.apply operated on th

당황
검증 실패: 0개
Empty DataFrame
Columns: [filename, errors]
Index: []

슬픔
검증 실패: 0개
Empty DataFrame
Columns: [filename, errors]
Index: []



/tmp/ipykernel_18013/1386583775.py:18: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled_data = data.groupby(["faceExp_uploader", "bg_uploader"]).apply(
